In [1]:
import nltk
from nltk.corpus import treebank
from collections import defaultdict

# =========================================================
# INITIALIZATION & DATASET SETUP
# =========================================================
# Uncomment the following lines if running for the first time
nltk.download('treebank')
nltk.download('universal_tagset')

print("========== POS TAGGING USING HMM & VITERBI ==========")

# Load the NLTK tagged corpus (Penn Treebank with Universal Tagset)
print("Loading NLTK Treebank tagged corpus...")
tagged_sentences = list(treebank.tagged_sents(tagset='universal'))

# Split into Training (95%) and Testing (5%) sets
split_index = int(len(tagged_sentences) * 0.95)
train_sents = tagged_sentences[:split_index]
# For quick execution in a flat script, we will test on a subset of 20 sentences
test_sents = tagged_sentences[split_index:split_index+20]

print(f"Total Sentences  : {len(tagged_sentences)}")
print(f"Training on      : {len(train_sents)} sentences")
print(f"Testing on       : {len(test_sents)} sentences\n")

# =========================================================
# 1. ESTIMATE TRANSITION AND EMISSION PROBABILITIES
# =========================================================
print("========== ESTIMATING PROBABILITIES ==========")

tags_count = defaultdict(int)
words_count = defaultdict(int)
emission_counts = defaultdict(lambda: defaultdict(int))
transition_counts = defaultdict(lambda: defaultdict(int))
start_counts = defaultdict(int)

# Process training data to get frequencies
for sent in train_sents:
    previous_tag = None
    for i, (word, tag) in enumerate(sent):
        word = word.lower()
        
        tags_count[tag] += 1
        words_count[word] += 1
        emission_counts[tag][word] += 1
        
        if i == 0:
            start_counts[tag] += 1
        else:
            transition_counts[previous_tag][tag] += 1
            
        previous_tag = tag

unique_tags = list(tags_count.keys())
num_sentences = len(train_sents)

print("Training complete. Extracted states (POS tags) and observations (words).\n")

# =========================================================
# 2. VITERBI ALGORITHM IMPLEMENTATION & EVALUATION
# =========================================================
print("========== VITERBI DECODING & EVALUATION ==========")

correct_predictions = 0
total_predictions = 0

# Small constant for smoothing to handle Out-Of-Vocabulary (OOV) words
SMOOTHING_FACTOR = 1e-6 

for sent_idx, sent in enumerate(test_sents):
    words = [w.lower() for w, t in sent]
    true_tags = [t for w, t in sent]
    
    viterbi_matrix = []
    backpointer_matrix = []
    
    # ---------------------------------------------------------
    # A. Initialization Step
    # ---------------------------------------------------------
    first_word = words[0]
    init_v = {}
    init_bp = {}
    
    for tag in unique_tags:
        # P(Start)
        start_prob = start_counts[tag] / num_sentences if start_counts[tag] > 0 else SMOOTHING_FACTOR
        # P(Word | Tag)
        emission_prob = emission_counts[tag][first_word] / tags_count[tag] if emission_counts[tag][first_word] > 0 else SMOOTHING_FACTOR
        
        init_v[tag] = start_prob * emission_prob
        init_bp[tag] = None # No previous state
        
    viterbi_matrix.append(init_v)
    backpointer_matrix.append(init_bp)
    
    # ---------------------------------------------------------
    # B. Recursion Step
    # ---------------------------------------------------------
    for t in range(1, len(words)):
        word = words[t]
        curr_v = {}
        curr_bp = {}
        
        for curr_tag in unique_tags:
            # Emission Probability
            emission_prob = emission_counts[curr_tag][word] / tags_count[curr_tag] if emission_counts[curr_tag][word] > 0 else SMOOTHING_FACTOR
            
            max_prob = -1
            best_prev_tag = None
            
            for prev_tag in unique_tags:
                # Transition Probability
                trans_prob = transition_counts[prev_tag][curr_tag] / tags_count[prev_tag] if transition_counts[prev_tag][curr_tag] > 0 else SMOOTHING_FACTOR
                
                # Viterbi Formula: v[t-1][prev_tag] * P(curr_tag | prev_tag) * P(word | curr_tag)
                prob = viterbi_matrix[t-1][prev_tag] * trans_prob * emission_prob
                
                if prob > max_prob:
                    max_prob = prob
                    best_prev_tag = prev_tag
                    
            curr_v[curr_tag] = max_prob
            curr_bp[curr_tag] = best_prev_tag
            
        viterbi_matrix.append(curr_v)
        backpointer_matrix.append(curr_bp)
        
    # ---------------------------------------------------------
    # C. Termination Step
    # ---------------------------------------------------------
    max_prob = -1
    best_last_tag = None
    
    for tag in unique_tags:
        if viterbi_matrix[-1][tag] > max_prob:
            max_prob = viterbi_matrix[-1][tag]
            best_last_tag = tag
            
    # ---------------------------------------------------------
    # D. Backtrack
    # ---------------------------------------------------------
    predicted_path = [best_last_tag]
    for t in range(len(words)-1, 0, -1):
        best_last_tag = backpointer_matrix[t][best_last_tag]
        predicted_path.insert(0, best_last_tag)
        
    # Track accuracy
    for p_tag, t_tag in zip(predicted_path, true_tags):
        if p_tag == t_tag:
            correct_predictions += 1
        total_predictions += 1
        
    # Print the first sentence as a demonstration
    if sent_idx == 0:
        print("Sample Viterbi Decoding Output:")
        print(f"{'WORD':<15} | {'TRUE TAG':<10} | {'PREDICTED TAG'}")
        print("-" * 45)
        for w, t_tag, p_tag in zip(words, true_tags, predicted_path):
            print(f"{w:<15} | {t_tag:<10} | {p_tag}")
        print("\nCalculating accuracy over remaining test sentences...\n")

# =========================================================
# 3. ACCURACY & ERROR ANALYSIS
# =========================================================
accuracy = (correct_predictions / total_predictions) * 100
print("========== FINAL RESULTS & ANALYSIS ==========")
print(f"Total Words Evaluated : {total_predictions}")
print(f"Correctly Tagged      : {correct_predictions}")
print(f"Tagging Accuracy      : {accuracy:.2f}%\n")

limitations_analysis = """
--- Error Analysis & Limitations of HMM ---
1. Out-Of-Vocabulary (OOV) Words: 
   HMMs struggle significantly with words not seen during training. In this script, we applied a basic smoothing factor (1e-6), but advanced morphological analysis is required to guess tags for unknown words accurately.
2. Markov Assumption Shortcomings:
   HMMs only look at the immediately preceding tag (Bigram assumption). They fail to capture long-distance syntactic dependencies in complex sentences (e.g., a subject at the beginning of a sentence dictating a verb far at the end).
3. Emission Independence:
   The model assumes the word generated depends strictly on the current tag, ignoring surrounding words. Modern neural models (like BERT) handle this much better by utilizing bidirectional context.
"""
print(limitations_analysis)

[nltk_data] Downloading package treebank to /home/mitu/nltk_data...
[nltk_data]   Unzipping corpora/treebank.zip.
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/mitu/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


========== POS TAGGING USING HMM & VITERBI ==========
Loading NLTK Treebank tagged corpus...
Total Sentences  : 3914
Training on      : 3718 sentences
Testing on       : 20 sentences

========== ESTIMATING PROBABILITIES ==========
Training complete. Extracted states (POS tags) and observations (words).

========== VITERBI DECODING & EVALUATION ==========
Sample Viterbi Decoding Output:
WORD            | TRUE TAG   | PREDICTED TAG
---------------------------------------------
inter-tel       | NOUN       | DET
inc             | NOUN       | NOUN
.               | .          | .
-lrb-           | .          | .
chandler        | NOUN       | NOUN
,               | .          | .
ariz.           | NOUN       | NOUN
-rrb-           | .          | .
--              | .          | .

Calculating accuracy over remaining test sentences...

========== FINAL RESULTS & ANALYSIS ==========
Total Words Evaluated : 568
Correctly Tagged      : 514
Tagging Accuracy      : 90.49%


--- Error Analysis &